## Cell 0 — Drive Setup and Imports

In [1]:
# Cell 0 — Drive Setup and Imports
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, glob, json, warnings, urllib.request
import numpy as np
import pandas as pd
import joblib
import stumpy

from numpy.linalg import pinv
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    precision_recall_curve,
)

warnings.filterwarnings('ignore')
RANDOMSTATE = 42
np.random.seed(RANDOMSTATE)

try:
    from numba import cuda
    USEGPU = cuda.is_available()
    print('GPU', 'Available' if USEGPU else 'Not available — CPU only')
except Exception:
    USEGPU = False
    print('GPU check failed — CPU only')

# Unified Drive paths
RUNID = 'ensemble_run_001'
DRIVEROOT = '/content/drive/MyDrive/tsad_ensemble_runs'
NOTEBOOKTAG = 'matrix_profile'

RUNDIR = os.path.join(DRIVEROOT, RUNID, NOTEBOOKTAG)
ARTIFACTDIR = os.path.join(RUNDIR, 'artifacts')
PREDICTIONSDIR = os.path.join(RUNDIR, 'predictions')
CACHEDIR = os.path.join(DRIVEROOT, '_cache')

for d in [ARTIFACTDIR, PREDICTIONSDIR, CACHEDIR]:
    os.makedirs(d, exist_ok=True)

print('RUNDIR       :', RUNDIR)
print('ARTIFACTDIR  :', ARTIFACTDIR)
print('PREDICTIONSDIR:', PREDICTIONSDIR)


Mounted at /content/drive
GPU Not available — CPU only
RUNDIR       : /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/matrix_profile
ARTIFACTDIR  : /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/matrix_profile/artifacts
PREDICTIONSDIR: /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/matrix_profile/predictions


## Cell 1 — Dataset Paths

In [2]:
# Cell 1 — Dataset Paths

MYDRIVE = '/content/drive/MyDrive'
CREDITCARDPATH = os.path.join(MYDRIVE, 'creditcard.csv')

NAB_CANDIDATES = [
    os.path.join(MYDRIVE, 'NAB Dataset'),
    os.path.join(MYDRIVE, 'NAB'),
    os.path.join(MYDRIVE, 'datasets', 'NAB Dataset'),
    os.path.join(MYDRIVE, 'datasets', 'NAB'),
]

def resolve_nab_root(candidates):
    for c in candidates:
        if os.path.isdir(c):
            csvs = glob.glob(os.path.join(c, '**', '*.csv'), recursive=True)
            if len(csvs) > 0:
                return c
    return None

NABROOT = resolve_nab_root(NAB_CANDIDATES)

# NAB labels
NABLABELSLOCAL = os.path.join(CACHEDIR, 'nab_combined_windows.json')
NABLABELSURL = 'https://raw.githubusercontent.com/numenta/NAB/master/labels/combined_windows.json'
if not os.path.exists(NABLABELSLOCAL):
    print('Downloading NAB labels...')
    urllib.request.urlretrieve(NABLABELSURL, NABLABELSLOCAL)

with open(NABLABELSLOCAL) as f:
    NABWINDOWSMAP = json.load(f)

assert os.path.exists(CREDITCARDPATH), f'creditcard.csv not found at {CREDITCARDPATH}'
assert NABROOT is not None, 'NAB root not found in Drive.'

nab_csvs = [f for f in glob.glob(os.path.join(NABROOT, '**', '*.csv'), recursive=True) if 'README' not in f]
print(f'CREDITCARDPATH: {CREDITCARDPATH}')
print(f'NABROOT       : {NABROOT}')
print(f'NAB CSV count : {len(nab_csvs)}')
print(f'NAB labels    : {len(NABWINDOWSMAP)} entries')


CREDITCARDPATH: /content/drive/MyDrive/creditcard.csv
NABROOT       : /content/drive/MyDrive/NAB Dataset
NAB CSV count : 58
NAB labels    : 58 entries


## Cell 2 — Shared Utilities

In [3]:
# Cell 2 — Shared Utilities

def robust_z(x, eps=1e-9, clip=50.0):
    x = np.asarray(x, dtype=np.float64)
    med = np.median(x)
    mad = np.median(np.abs(x - med))
    return np.clip((x - med) / max(1.4826 * mad, eps), -clip, clip)

def keep_runs(y, min_len=3):
    y = np.asarray(y, dtype=np.int8).copy()
    n = len(y)
    i = 0
    while i < n:
        if y[i] == 1:
            j = i
            while j < n and y[j] == 1:
                j += 1
            if (j - i) < min_len:
                y[i:j] = 0
            i = j
        else:
            i += 1
    return y

def point_adjust(y_true, y_pred):
    yt = np.asarray(y_true, dtype=np.int8)
    yp = np.asarray(y_pred, dtype=np.int8).copy()
    n = len(yt)
    i = 0
    while i < n:
        if yt[i] == 1:
            j = i
            while j < n and yt[j] == 1:
                j += 1
            if yp[i:j].any():
                yp[i:j] = 1
            i = j
        else:
            i += 1
    return yp

def rolling_max_1d(x, w):
    x = np.asarray(x, dtype=np.float64)
    if w <= 1:
        return x
    return pd.Series(x).rolling(w, min_periods=1).max().values

def best_strict_f1_threshold(y, s, ngrid=300, qlo=0.50, qhi=0.999, minrun=3):
    y = np.asarray(y, dtype=int)
    s = np.asarray(s, dtype=float)
    if y.sum() == 0:
        return float(np.max(s)) if len(s) else 0.0, 0.0
    qs = np.linspace(qlo, qhi, ngrid)
    thr_list = np.unique(np.quantile(s, qs))
    best_t, best_f = float(thr_list[-1]), -1.0
    for t in thr_list:
        p = keep_runs((s >= t).astype(int), min_len=minrun)
        f = f1_score(y, p, zero_division=0)
        if f > best_f:
            best_f, best_t = float(f), float(t)
    return best_t, best_f

def compute_metrics(y, pred, scores=None, prefix=''):
    m = {
        f'{prefix}precision': float(precision_score(y, pred, zero_division=0)),
        f'{prefix}recall': float(recall_score(y, pred, zero_division=0)),
        f'{prefix}f1': float(f1_score(y, pred, zero_division=0)),
    }
    if scores is not None and len(np.unique(y)) == 2:
        m[f'{prefix}rocauc'] = float(roc_auc_score(y, scores))
        m[f'{prefix}prauc'] = float(average_precision_score(y, scores))
    else:
        m[f'{prefix}rocauc'] = float('nan')
        m[f'{prefix}prauc'] = float('nan')
    return m

print('Utilities ready.')


Utilities ready.


In [4]:
# DEBUG — Verify utility functions work correctly
print('=== UTILITY FUNCTION VERIFICATION ===')

# Test keep_runs
test_y = np.array([0,0,1,1,0,0,1,0,1,1,1,1,0], dtype=np.int8)
result = keep_runs(test_y, min_len=3)
print(f'keep_runs input : {test_y.tolist()}')
print(f'keep_runs output: {result.tolist()}')
assert result[2:4].sum() == 0, 'Short run of 2 should be removed'
assert result[8:12].sum() == 4, 'Run of 4 should be kept'
print('keep_runs: PASS')

# Test best_strict_f1_threshold with known data
test_ytrue = np.array([0]*90 + [1]*10)
test_scores = np.array([0.1]*85 + [0.5]*5 + [0.8]*7 + [0.3]*3)
thr, f1 = best_strict_f1_threshold(test_ytrue, test_scores, ngrid=50, minrun=1)
print(f'best_strict_f1_threshold: thr={thr:.4f}, f1={f1:.4f}')
assert f1 > 0.0, 'Should find nonzero F1'
print('best_strict_f1_threshold: PASS (no self error, no minlen error)')

# Test point_adjust
pa = point_adjust(test_ytrue, (test_scores > 0.7).astype(np.int8))
print(f'point_adjust: detected {pa.sum()} vs true {test_ytrue.sum()}')
print('point_adjust: PASS')
print()
print('All utility functions verified.')


=== UTILITY FUNCTION VERIFICATION ===
keep_runs input : [0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0]
keep_runs output: [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0]
keep_runs: PASS
best_strict_f1_threshold: thr=0.7556, f1=0.8235
best_strict_f1_threshold: PASS (no self error, no minlen error)
point_adjust: detected 10 vs true 10
point_adjust: PASS

All utility functions verified.


## Cell 3 — Credit Card Matrix Profile Agent

Uses Mahalanobis distance on the V-features + log(Amount) as an anomaly score.
Robust covariance estimation with outlier trimming. PR-curve threshold tuning
on validation set. Strict holdout on 20% test set (same split as Memto/IF).

Entity ID = `creditcard` for coordinator compatibility.


In [5]:
# Cell 3 — Credit Card MP Agent

class CreditCardMPAgent:
    TOPK = 29
    OUTLIERFRAC = 0.005
    RANDOMSTATE = 42

    def __init__(self):
        self.topfeats = None
        self.mu = None
        self.covinv = None
        self.threshold = None
        self.featurecols = None
        self.testindices = None

    def load(self):
        df = pd.read_csv(CREDITCARDPATH).dropna()
        df = df.sort_values('Time').reset_index(drop=True)
        df['Amountlog'] = np.log1p(df['Amount'].astype(float))
        featurecols = [f'V{i}' for i in range(1, 29)] + ['Amountlog']
        X = df[featurecols].values.astype(np.float64)
        y = df['Class'].astype(int).values
        self.featurecols = featurecols
        return X, y

    def score(self, X):
        d = X - self.mu
        s = np.einsum('ij,jk,ik->i', d, self.covinv, d)
        s = np.sqrt(np.clip(np.nan_to_num(s, nan=0.0, posinf=0.0, neginf=0.0), 0.0, None))
        return s.astype(np.float64)

    def fit(self):
        Xall, yall = self.load()
        Xnormal = Xall[yall == 0]
        print(f'CC-MP rows {len(Xall):,} frauds {int(yall.sum())} rate {100*yall.mean():.4f}%')

        self.topfeats = np.arange(Xnormal.shape[1])[:self.TOPK]
        Xn = Xnormal[:, self.topfeats]

        # Robust fit: compute rough Mahalanobis, trim outliers, refit
        mu_rough = Xn.mean(axis=0)
        cov_rough = np.cov(Xn.T) + 1e-6 * np.eye(Xn.shape[1])
        cinv_rough = pinv(cov_rough)
        d_rough = Xn - mu_rough
        s_rough = np.sqrt(np.clip(np.einsum('ij,jk,ik->i', d_rough, cinv_rough, d_rough), 0.0, None))

        cutoff = np.percentile(s_rough, (1 - self.OUTLIERFRAC) * 100)
        Xclean = Xn[s_rough <= cutoff]
        print(f'CC-MP robust fit: kept {len(Xclean):,}/{len(Xn):,} normal rows')

        self.mu = Xclean.mean(axis=0)
        cov = np.cov(Xclean.T) + 1e-6 * np.eye(Xclean.shape[1])
        self.covinv = pinv(cov)
        return self

    def evaluate(self):
        Xall, yall = self.load()
        scores = self.score(Xall[:, self.topfeats])

        idx = np.arange(len(yall), dtype=np.int64)
        # Same split as Memto and IF: 80/20 with stratification
        idxtv, idxtest = train_test_split(
            idx, test_size=0.20, stratify=yall, random_state=self.RANDOMSTATE
        )
        # Further split tune into fit/val for threshold tuning
        idxfit, idxval = train_test_split(
            idxtv, test_size=0.25, stratify=yall[idxtv], random_state=self.RANDOMSTATE
        )

        # Threshold from PR-curve on validation
        prec, rec, thr = precision_recall_curve(yall[idxval], scores[idxval])
        f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
        if len(thr) == 0:
            self.threshold = float(np.percentile(scores[idxval], 99.5))
        else:
            self.threshold = float(thr[int(np.nanargmax(f1s))])

        ypredtest = (scores[idxtest] >= self.threshold).astype(np.int8)
        self.testindices = idxtest.astype(np.int64).copy()

        strict = compute_metrics(yall[idxtest], ypredtest, scores[idxtest])

        results = {
            'dataset': 'creditcard',
            'protocol': 'strict point-wise holdout (20% test)',
            **strict,
            'threshold': float(self.threshold),
            'n_anomalies_true': int(yall[idxtest].sum()),
            'n_anomalies_pred': int(ypredtest.sum()),
        }

        return (
            results,
            scores[idxtest].astype(np.float32),
            yall[idxtest].astype(np.int8),
            ypredtest.astype(np.int8),
        )

    def save_artifact(self):
        path = os.path.join(ARTIFACTDIR, 'mpcreditcard.joblib')
        joblib.dump({
            'topfeats': self.topfeats,
            'mu': self.mu,
            'covinv': self.covinv,
            'threshold': self.threshold,
            'featurecols': self.featurecols,
            'testindices': self.testindices,
        }, path)
        print('Saved artifact ->', path)

ccmpagent = CreditCardMPAgent()
print('CC-MP fitting...')
ccmpagent.fit()

print('CC-MP evaluating...')
ccmpresults, ccmpscores, ccmpytrue, ccmppred = ccmpagent.evaluate()
ccmpagent.save_artifact()

print()
print('=' * 60)
print('CREDIT CARD MATRIX PROFILE RESULTS')
print('=' * 60)
for k, v in ccmpresults.items():
    print(f'  {k:25s}: {v:.6f}' if isinstance(v, float) else f'  {k:25s}: {v}')


CC-MP fitting...
CC-MP rows 284,807 frauds 492 rate 0.1727%
CC-MP robust fit: kept 282,893/284,315 normal rows
CC-MP evaluating...
Saved artifact -> /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/matrix_profile/artifacts/mpcreditcard.joblib

CREDIT CARD MATRIX PROFILE RESULTS
  dataset                  : creditcard
  protocol                 : strict point-wise holdout (20% test)
  precision                : 0.821053
  recall                   : 0.795918
  f1                       : 0.808290
  rocauc                   : 0.964591
  prauc                    : 0.695749
  threshold                : 302.642458
  n_anomalies_true         : 98
  n_anomalies_pred         : 95


In [6]:
# DEBUG — CC Matrix Profile diagnostics
print('=== CC MATRIX PROFILE DIAGNOSTICS ===')
print(f'Threshold: {ccmpagent.threshold:.4f}')
print(f'Top features used: {ccmpagent.topfeats[:5]}...')
print(f'Covariance inverse shape: {ccmpagent.covinv.shape}')
print(f'Test set: {len(ccmpytrue)} samples, {int(ccmpytrue.sum())} frauds')
print(f'Predictions: {int(ccmppred.sum())} positive')
print(f'Score range: [{ccmpscores.min():.4f}, {ccmpscores.max():.4f}]')
print(f'Score at threshold percentile: {(ccmpscores < ccmpagent.threshold).mean()*100:.2f}%')
# Check FP/FN breakdown
fp = int(((ccmppred == 1) & (ccmpytrue == 0)).sum())
fn = int(((ccmppred == 0) & (ccmpytrue == 1)).sum())
tp = int(((ccmppred == 1) & (ccmpytrue == 1)).sum())
tn = int(((ccmppred == 0) & (ccmpytrue == 0)).sum())
print(f'Confusion: TP={tp} FP={fp} FN={fn} TN={tn}')
print(f'Precision={tp/(tp+fp+1e-9):.4f} Recall={tp/(tp+fn+1e-9):.4f}')


=== CC MATRIX PROFILE DIAGNOSTICS ===
Threshold: 302.6425
Top features used: [0 1 2 3 4]...
Covariance inverse shape: (29, 29)
Test set: 56962 samples, 98 frauds
Predictions: 95 positive
Score range: [2.2483, 1450.6675]
Score at threshold percentile: 99.83%
Confusion: TP=78 FP=17 FN=20 TN=56847
Precision=0.8211 Recall=0.7959


## Cell 4 — NAB Matrix Profile Agent

Uses a fusion of three anomaly signals per series:
1. Rolling median absolute deviation (change detection)
2. First-order differences (spike detection)
3. Matrix profile discord score (shape-based anomaly)

Three-way temporal split: train 40%, validate 30%, test 30%.
Threshold tuned on validation using strict F1 grid search.
Reports both strict and point-adjusted metrics.
Per-category breakdown in the next cell.

Entity IDs use basenames for coordinator alignment with IF.


In [7]:
# Cell 4 — NAB MP Agent

class NABMPAgent:
    WINDOW_FRAC = 0.02
    MIN_WINDOW = 20
    MAX_WINDOW = 100

    W_ROLLING = 0.60
    W_DIFF = 0.35
    W_MP = 0.50

    MIN_RUN = 3
    THR_GRID = 300
    TRAINFRAC = 0.40
    VALFRAC = 0.30

    def __init__(self):
        self.series_params = {}

    def _load_nab_csv(self, path):
        df = pd.read_csv(path)
        ts = pd.to_datetime(df.iloc[:, 0])
        vals = df.iloc[:, 1].values.astype(np.float64)
        return vals, ts

    def _match_nab_key(self, csv_path):
        rel = os.path.relpath(csv_path, NABROOT).replace('\\', '/')
        if rel in NABWINDOWSMAP:
            return rel
        base = os.path.basename(csv_path)
        for k in NABWINDOWSMAP:
            if os.path.basename(k) == base:
                return k
        return None

    def _make_labels(self, ts, csv_path):
        y = np.zeros(len(ts), dtype=np.int8)
        key = self._match_nab_key(csv_path)
        if key is not None and NABWINDOWSMAP.get(key):
            for t0, t1 in NABWINDOWSMAP[key]:
                y[(ts >= pd.Timestamp(t0)) & (ts <= pd.Timestamp(t1))] = 1
        return y

    def _score_series(self, vals, window):
        vals = np.nan_to_num(np.asarray(vals, dtype=np.float64), nan=0.0)
        n = len(vals)
        w = int(max(self.MIN_WINDOW, min(self.MAX_WINDOW, window)))
        if n <= w + 1:
            return np.zeros(n, dtype=np.float64)

        s = pd.Series(vals)

        # Signal 1: rolling MAD
        roll_med = s.rolling(w, min_periods=1).median().values
        roll_mad = (s - pd.Series(roll_med)).abs().rolling(w, min_periods=1).median().values
        roll_mad = np.maximum(roll_mad, 1e-9)
        ch_mad = np.abs(vals - roll_med) / (roll_mad * 1.4826)

        # Signal 2: first-order differences
        diff = np.abs(np.diff(vals, prepend=vals[0]))
        diff_mad = np.median(np.abs(diff - np.median(diff))) + 1e-9
        ch_diff = diff / (diff_mad * 1.4826)

        # Signal 3: matrix profile discord
        col = np.ascontiguousarray(vals, dtype=np.float64)
        try:
            if USEGPU:
                mp = stumpy.gpu_stump(col, m=w)[:, 0].astype(np.float64)
            else:
                mp = stumpy.stump(col, m=w)[:, 0].astype(np.float64)
        except Exception:
            mp = stumpy.stump(col, m=w)[:, 0].astype(np.float64)

        fin = mp[np.isfinite(mp)]
        fill = float(fin.max()) if len(fin) else 0.0
        mp = np.nan_to_num(mp, nan=fill, posinf=fill, neginf=0.0)
        mp_full = np.concatenate([mp, np.full(w - 1, mp[-1], dtype=np.float64)])
        ch_mp = rolling_max_1d(mp_full, w)

        def _norm(a):
            lo, hi = float(np.min(a)), float(np.max(a))
            return (a - lo) / (hi - lo + 1e-9)

        score = np.maximum.reduce([
            self.W_ROLLING * _norm(ch_mad),
            self.W_DIFF * _norm(ch_diff),
            self.W_MP * _norm(ch_mp),
        ])
        return score.astype(np.float64)

    def fit(self):
        csv_files = [f for f in glob.glob(os.path.join(NABROOT, '**', '*.csv'), recursive=True)
                     if os.path.isfile(f) and 'README' not in f]
        print(f'[NAB-MP] found {len(csv_files)} csv files')
        for csv_path in sorted(csv_files):
            try:
                vals, _ = self._load_nab_csv(csv_path)
                n = len(vals)
                w = int(max(self.MIN_WINDOW, min(self.MAX_WINDOW, n * self.WINDOW_FRAC)))
                key = os.path.relpath(csv_path, NABROOT).replace('\\', '/')
                self.series_params[key] = {'csv_path': csv_path, 'window': w}
            except Exception as e:
                print(f'[NAB-MP] fit skip: {os.path.basename(csv_path)} -> {e}')
        print(f'[NAB-MP] fitted {len(self.series_params)} series')
        return self

    def evaluate(self):
        bundle = {}
        S_all, Y_all, P_all, PA_all = [], [], [], []
        category_data = {}

        for i, (key, params) in enumerate(sorted(self.series_params.items())):
            try:
                vals, ts = self._load_nab_csv(params['csv_path'])
                y = self._make_labels(ts, params['csv_path'])
                raw_score = self._score_series(vals, params['window'])
                sz = robust_z(raw_score)

                n = len(y)
                n_tr = int(round(n * self.TRAINFRAC))
                n_val = int(round(n * self.VALFRAC))
                sl_val = slice(n_tr, n_tr + n_val)
                sl_test = slice(n_tr + n_val, n)

                if sl_test.stop <= sl_test.start:
                    continue

                # Threshold on validation portion
                if y[sl_val].sum() == 0:
                    thr = float(np.percentile(sz[sl_val], 99.5))
                    val_f1 = 0.0
                else:
                    thr, val_f1 = best_strict_f1_threshold(
                        y[sl_val], sz[sl_val], ngrid=self.THR_GRID, minrun=self.MIN_RUN
                    )

                pred_full = keep_runs((sz >= thr).astype(np.int8), min_len=self.MIN_RUN)

                y_h = y[sl_test]
                s_h = sz[sl_test]
                p_h = pred_full[sl_test]
                pa_h = point_adjust(y_h, p_h)

                # Use basename as entity ID for coordinator
                entity_name = os.path.basename(key)

                bundle[entity_name] = {
                    'entity_id': entity_name,
                    'nab_key': key,
                    'scores_full': sz.astype(np.float32),
                    'y_full': y.astype(np.int8),
                    'pred_full': pred_full.astype(np.int8),
                    'row_id': np.arange(len(y), dtype=np.int64),
                    'val_slice': (sl_val.start, sl_val.stop),
                    'test_slice': (sl_test.start, sl_test.stop),
                    'threshold': float(thr),
                    'val_f1': float(val_f1),
                }

                S_all.append(s_h)
                Y_all.append(y_h)
                P_all.append(p_h)
                PA_all.append(pa_h)

                cat = key.split('/')[0] if '/' in key else 'unknown'
                if cat not in category_data:
                    category_data[cat] = {'S': [], 'Y': [], 'P': [], 'PA': []}
                category_data[cat]['S'].append(s_h)
                category_data[cat]['Y'].append(y_h)
                category_data[cat]['P'].append(p_h)
                category_data[cat]['PA'].append(pa_h)

                if i < 3:
                    print(f'  [debug] {key}: n={len(y)} true={int(y.sum())} pred={int(pred_full.sum())}')

            except Exception as e:
                print(f'[NAB-MP] eval skip {key}: {e}')

        if not S_all:
            return {}, bundle, category_data

        S = np.concatenate(S_all).astype(np.float32)
        Y = np.concatenate(Y_all).astype(np.int8)
        P = np.concatenate(P_all).astype(np.int8)
        PA = np.concatenate(PA_all).astype(np.int8)

        strict = compute_metrics(Y, P, S, prefix='strict_')
        pa_m = compute_metrics(Y, PA, S, prefix='pa_')

        results = {
            'dataset': 'nab',
            'protocol': f'temporal holdout: train {int(100*self.TRAINFRAC)}% / val {int(100*self.VALFRAC)}% / test {int(100*(1-self.TRAINFRAC-self.VALFRAC))}%',
            **strict,
            **pa_m,
            'n_anomalies_true': int(Y.sum()),
            'n_anomalies_pred': int(P.sum()),
            'n_series': len(bundle),
        }
        return results, bundle, category_data

    def save(self, bundle):
        path = os.path.join(PREDICTIONSDIR, 'mpnabstrictpointwise.joblib')
        joblib.dump({
            'dataset': 'nab',
            'model': 'matrix_profile',
            'protocol': 'temporal holdout strict + PA',
            'entities': bundle,
        }, path)
        print('[NAB-MP] Saved', path)

nabmpagent = NABMPAgent()
print('NAB-MP fitting...')
nabmpagent.fit()

print('NAB-MP evaluating...')
nabmpresults, nabmpbundle, nabmp_cat_data = nabmpagent.evaluate()
nabmpagent.save(nabmpbundle)

print()
print('=' * 60)
print('NAB MATRIX PROFILE RESULTS (GLOBAL)')
print('=' * 60)
for k, v in nabmpresults.items():
    print(f'  {k:25s}: {v:.6f}' if isinstance(v, float) else f'  {k:25s}: {v}')


NAB-MP fitting...
[NAB-MP] found 58 csv files
[NAB-MP] fitted 58 series
NAB-MP evaluating...
  [debug] artificialNoAnomaly/art_daily_no_noise.csv: n=4032 true=0 pred=619
  [debug] artificialNoAnomaly/art_daily_perfect_square_wave.csv: n=4032 true=0 pred=1092
  [debug] artificialNoAnomaly/art_daily_small_noise.csv: n=4032 true=0 pred=3
[NAB-MP] Saved /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/matrix_profile/predictions/mpnabstrictpointwise.joblib

NAB MATRIX PROFILE RESULTS (GLOBAL)
  dataset                  : nab
  protocol                 : temporal holdout: train 40% / val 30% / test 30%
  strict_precision         : 0.097257
  strict_recall            : 0.229304
  strict_f1                : 0.136584
  strict_rocauc            : 0.525601
  strict_prauc             : 0.207304
  pa_precision             : 0.204625
  pa_recall                : 0.547571
  pa_f1                    : 0.297918
  pa_rocauc                : 0.525601
  pa_prauc                 : 0.207304
  n_an

In [8]:
# DEBUG — NAB Matrix Profile diagnostics
print('=== NAB MATRIX PROFILE DIAGNOSTICS ===')
print(f'Total series fitted: {len(nabmpagent.series_params)}')
print(f'Total series evaluated: {nabmpresults.get("n_series", 0)}')
if nabmpresults.get('n_series', 0) < 58:
    print(f'WARNING: Only {nabmpresults.get("n_series", 0)}/58 series evaluated!')
    print('Check for errors in evaluate() — likely parameter name mismatch or runtime error')
print(f'True anomalies: {nabmpresults.get("n_anomalies_true", 0)}')
print(f'Pred anomalies: {nabmpresults.get("n_anomalies_pred", 0)}')
print(f'NAB bundle entities: {len(nabmpbundle)}')
# Sample entity diagnostics
for i, (eid, ent) in enumerate(sorted(nabmpbundle.items())[:3]):
    print(f'  Entity {eid}: scores shape={ent["scores_full"].shape}, '
          f'y_sum={ent["y_full"].sum()}, pred_sum={ent["pred_full"].sum()}')
print()
# Category coverage
cats = {}
for key in nabmpagent.series_params:
    cat = key.split('/')[0] if '/' in key else 'unknown'
    cats[cat] = cats.get(cat, 0) + 1
print('Series per category:')
for cat, count in sorted(cats.items()):
    in_bundle = sum(1 for k in nabmpbundle if cat in nabmpagent.series_params.get(k, {}).get('csv_path', ''))
    print(f'  {cat}: {count} fitted')


=== NAB MATRIX PROFILE DIAGNOSTICS ===
Total series fitted: 58
Total series evaluated: 58
True anomalies: 11814
Pred anomalies: 27854
NAB bundle entities: 58
  Entity TravelTime_387.csv: scores shape=(2500,), y_sum=249, pred_sum=1541
  Entity TravelTime_451.csv: scores shape=(2162,), y_sum=217, pred_sum=373
  Entity Twitter_volume_AAPL.csv: scores shape=(15902,), y_sum=1588, pred_sum=3381

Series per category:
  artificialNoAnomaly: 5 fitted
  artificialWithAnomaly: 6 fitted
  realAWSCloudwatch: 17 fitted
  realAdExchange: 6 fitted
  realKnownCause: 7 fitted
  realTraffic: 7 fitted
  realTweets: 10 fitted


## Cell 5 — NAB Per-Category Breakdown

In [9]:
# Cell 5 — NAB Per-Category Breakdown

print('NAB MP PER-CATEGORY RESULTS')
print('=' * 90)

cat_rows = []
for cat, data in sorted(nabmp_cat_data.items()):
    Y = np.concatenate(data['Y'])
    P = np.concatenate(data['P'])
    S = np.concatenate(data['S'])
    PA = np.concatenate(data['PA'])

    strict = compute_metrics(Y, P, S)
    pa = compute_metrics(Y, PA, S, prefix='pa_')

    row = {
        'category': cat,
        'n_series': len(data['Y']),
        'n_points': len(Y),
        'n_anomalies': int(Y.sum()),
        'anomaly_rate': float(Y.mean()),
        **strict,
        **pa,
    }
    cat_rows.append(row)
    print(f"  {cat:30s}  series={row['n_series']:3d}  "
          f"strict F1={strict['f1']:.4f}  PA F1={pa['pa_f1']:.4f}  "
          f"anomaly_rate={row['anomaly_rate']:.4f}")

nabmp_cat_df = pd.DataFrame(cat_rows)
display(nabmp_cat_df)


NAB MP PER-CATEGORY RESULTS
  artificialNoAnomaly             series=  5  strict F1=0.0000  PA F1=0.0000  anomaly_rate=0.0000
  artificialWithAnomaly           series=  6  strict F1=0.2950  PA F1=0.5266  anomaly_rate=0.2308
  realAWSCloudwatch               series= 17  strict F1=0.1165  PA F1=0.2416  anomaly_rate=0.0943
  realAdExchange                  series=  6  strict F1=0.2091  PA F1=0.2369  anomaly_rate=0.0898
  realKnownCause                  series=  7  strict F1=0.1804  PA F1=0.3398  anomaly_rate=0.1873
  realTraffic                     series=  7  strict F1=0.0031  PA F1=0.1227  anomaly_rate=0.2160
  realTweets                      series= 10  strict F1=0.1083  PA F1=0.2890  anomaly_rate=0.0639


,category,n_series,n_points,n_anomalies,anomaly_rate,precision,recall,f1,rocauc,prauc,pa_precision,pa_recall,pa_f1,pa_rocauc,pa_prauc
0,artificialNoAnomaly,5,6045,0,0.000000,0.000000,0.000000,0.000000,NaN,NaN,0.000000,0.000000,0.000000,NaN,NaN
1,artificialWithAnomaly,6,7254,1674,0.230769,0.230640,0.409200,0.295004,0.570917,0.381553,0.382432,0.845281,0.526610,0.570917,0.381553
2,realAWSCloudwatch,17,20315,1916,0.094315,0.083371,0.193111,0.116462,0.485909,0.234037,0.168098,0.429019,0.241552,0.485909,0.234037
3,realAdExchange,6,2884,259,0.089806,0.140397,0.409266,0.209073,0.462375,0.097041,0.158236,0.471042,0.236893,0.462375,0.097041
4,realKnownCause,7,20868,3908,0.187272,0.197444,0.166070,0.180403,0.661597,0.479345,0.336853,0.342886,0.339843,0.661597,0.479345
5,realTraffic,7,4699,1015,0.216003,0.003333,0.002956,0.003133,0.279410,0.154523,0.122309,0.123153,0.122730,0.279410,0.154523
6,realTweets,10,47590,3042,0.063921,0.066311,0.294543,0.108252,0.447261,0.068193,0.173318,0.869494,0.289024,0.447261,0.068193


## Cell 6 — Export for Coordinator

In [10]:
# Cell 6 — Export for Coordinator

exported = {}
summaryrows = []

# --- Credit Card ---
ccbundle = {
    'dataset': 'creditcard',
    'model': 'matrixprofile',
    'protocol': 'strict point-wise holdout',
    'entities': {
        'creditcard': {
            'entityid': 'creditcard',
            'scoresfull': ccmpscores,
            'yfull': ccmpytrue,
            'predfull': ccmppred,
            'rowid': np.arange(len(ccmpytrue), dtype=np.int64),
            'originalrowid': ccmpagent.testindices.astype(np.int64),
            'threshold': float(ccmpagent.threshold),
        }
    }
}

ccpath = os.path.join(PREDICTIONSDIR, 'mpcreditcardstrictpointwise.joblib')
joblib.dump(ccbundle, ccpath)
exported['creditcard'] = ccpath
summaryrows.append({'dataset': 'creditcard', **{k: v for k, v in ccmpresults.items() if k != 'dataset'}})
print('Saved', ccpath)

# --- NAB ---
nabpath = os.path.join(PREDICTIONSDIR, 'mpnabstrictpointwise.joblib')
# Already saved in Cell 4, just register
exported['nab'] = nabpath
summaryrows.append({'dataset': 'nab', **{k: v for k, v in nabmpresults.items() if k != 'dataset'}})

# --- Summary ---
summary = pd.DataFrame(summaryrows)
summarypath = os.path.join(PREDICTIONSDIR, 'mpstrictsummary.csv')
summary.to_csv(summarypath, index=False)
print('Saved', summarypath)
display(summary)

# --- Manifest ---
manifest = {
    'runid': RUNID,
    'driveroot': DRIVEROOT,
    'notebooktag': NOTEBOOKTAG,
    'modelfamily': 'matrixprofile',
    'exportprotocol': 'ensembleexportv2',
    'artifactsdir': ARTIFACTDIR,
    'predictionsdir': PREDICTIONSDIR,
    'exports': exported,
    'summarycsv': summarypath,
    'creditcard_originalrowid_included': True,
    'datasets': ['creditcard', 'nab'],
    'smd_dropped': True,
}

manifestpath = os.path.join(PREDICTIONSDIR, 'mpmanifest.json')
with open(manifestpath, 'w') as f:
    json.dump(manifest, f, indent=2)
print('Saved', manifestpath)

print()
print('Coordinator-ready files in:', PREDICTIONSDIR)
print('Exported datasets:', sorted(exported.keys()))


Saved /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/matrix_profile/predictions/mpcreditcardstrictpointwise.joblib
Saved /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/matrix_profile/predictions/mpstrictsummary.csv


,dataset,protocol,precision,recall,f1,rocauc,prauc,threshold,n_anomalies_true,n_anomalies_pred,...,strict_recall,strict_f1,strict_rocauc,strict_prauc,pa_precision,pa_recall,pa_f1,pa_rocauc,pa_prauc,n_series
0,creditcard,strict point-wise holdout (20% test),0.821053,0.795918,0.80829,0.964591,0.695749,302.642458,98,95,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,nab,temporal holdout: train 40% / val 30% / test 30%,NaN,NaN,NaN,NaN,NaN,NaN,11814,27854,...,0.229304,0.136584,0.525601,0.207304,0.204625,0.547571,0.297918,0.525601,0.207304,58.0


Saved /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/matrix_profile/predictions/mpmanifest.json

Coordinator-ready files in: /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/matrix_profile/predictions
Exported datasets: ['creditcard', 'nab']


## Cell 7 — Final Summary

In [11]:
# Cell 7 — Final Summary

print('=' * 70)
print('MATRIX PROFILE FINAL SUMMARY')
print('=' * 70)

print()
print('CREDIT CARD:')
for k, v in ccmpresults.items():
    print(f'  {k:25s}: {v:.6f}' if isinstance(v, float) else f'  {k:25s}: {v}')

print()
print('NAB (global):')
for k, v in nabmpresults.items():
    print(f'  {k:25s}: {v:.6f}' if isinstance(v, float) else f'  {k:25s}: {v}')

print()
print('NAB (per-category):')
display(nabmp_cat_df[['category', 'n_series', 'n_anomalies', 'f1', 'pa_f1', 'rocauc']].round(4))

print()
print('All exports saved to:', PREDICTIONSDIR)
print('Done.')


MATRIX PROFILE FINAL SUMMARY

CREDIT CARD:
  dataset                  : creditcard
  protocol                 : strict point-wise holdout (20% test)
  precision                : 0.821053
  recall                   : 0.795918
  f1                       : 0.808290
  rocauc                   : 0.964591
  prauc                    : 0.695749
  threshold                : 302.642458
  n_anomalies_true         : 98
  n_anomalies_pred         : 95

NAB (global):
  dataset                  : nab
  protocol                 : temporal holdout: train 40% / val 30% / test 30%
  strict_precision         : 0.097257
  strict_recall            : 0.229304
  strict_f1                : 0.136584
  strict_rocauc            : 0.525601
  strict_prauc             : 0.207304
  pa_precision             : 0.204625
  pa_recall                : 0.547571
  pa_f1                    : 0.297918
  pa_rocauc                : 0.525601
  pa_prauc                 : 0.207304
  n_anomalies_true         : 11814
  n_anomalies_pr

,category,n_series,n_anomalies,f1,pa_f1,rocauc
0,artificialNoAnomaly,5,0,0.0000,0.0000,NaN
1,artificialWithAnomaly,6,1674,0.2950,0.5266,0.5709
2,realAWSCloudwatch,17,1916,0.1165,0.2416,0.4859
3,realAdExchange,6,259,0.2091,0.2369,0.4624
4,realKnownCause,7,3908,0.1804,0.3398,0.6616
5,realTraffic,7,1015,0.0031,0.1227,0.2794
6,realTweets,10,3042,0.1083,0.2890,0.4473



All exports saved to: /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/matrix_profile/predictions
Done.
